In [0]:
%run ./00_utils

In [0]:
%pip install pandarallel

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window

import warnings
import numpy as np
import pandas as pd
import mlflow
from pandarallel import pandarallel

pandarallel.initialize(nb_workers = 16, progress_bar=False)

# 1. Data Preparation

In [0]:
CUTOFF_DAYS = 7
CUTOFF_NUM_OCCURRENCES = 10
MIN_COMPETITOR_OVERLAP = 0.6
CATALOG = get_catalog()
TABLE_STORICO_PROGRAMMI = f"{CATALOG}.whatif.storico_programmi"

print('Reading df_programmi...')
df_programmi = read_df_programmi(table_name=TABLE_STORICO_PROGRAMMI) # pandas dataframe

print('Computing precise historical stats...')
df_programmi = df_programmi.groupby(['Programma','Ora','Canale','GiornoSettimana']).parallel_apply(
    compute_stats_last_occurrences, 
    suffix='Precise', 
    cutoff_days=CUTOFF_DAYS, 
    cutoff_num_occurrences=CUTOFF_NUM_OCCURRENCES).reset_index(drop=True)

print('Computing previous and next program stats...')
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    df_programmi = df_programmi.groupby(['Canale']).parallel_apply(
        compute_stats_prev_next_program, 
        suffix='Precise').reset_index(drop=True)

df_programmi_pre = spark.createDataFrame(df_programmi)
df_programmi_pre = df_programmi_pre.where(F.col('StoricoSharePrecise') >= 0.0)

print('Done!')

In [0]:
df_programmi_pre = add_competitor(
    df_programmi_pre, 
    suffix='Precise', 
    min_competitor_overlap=MIN_COMPETITOR_OVERLAP)

for c in df_programmi_pre.columns:
    if c.endswith('Precise'):
        df_programmi_pre = df_programmi_pre.withColumnRenamed(c, c.replace('Precise',''))

In [0]:
df_programmi = df_programmi_pre

df_programmi = df_programmi.withColumn('ShareResiduo', F.col('Share') - F.col('StoricoShare'))
df_programmi = df_programmi.withColumn('HighValueShare', F.col('StoricoShare') >= 0.12)
df_programmi = df_programmi.withColumn('Programma_durata', F.col("ORA_FINE_TRX") - F.col("ORA_INIZIO_TRX"))

for c in ['StoricoShareMax','StoricoShareMin','StoricoShareLast','StoricoShare_prev','StoricoShare_next','StoricoShareLast_prev','StoricoShareLast_next']:
    df_programmi = df_programmi.withColumn(c, F.col(c) - F.col('StoricoShare'))

df_programmi = (
    df_programmi
    .withColumn('Ora', F.col('Ora').cast('string'))
    .withColumn('GiornoSettimana', F.col('GiornoSettimana').cast('string'))
    .withColumn('Mese', F.col('Mese').cast('string'))
)

df_programmi = (
    df_programmi.withColumn('StoricoShareUomini_delta', F.col('StoricoShareUomini') * F.col('StoricoShare') - F.col('StoricoShareUomini_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_08_14_delta', F.col('StoricoShare_08_14') * F.col('StoricoShare') - F.col('StoricoShare_08_14_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_15_24_delta', F.col('StoricoShare_15_24') * F.col('StoricoShare') - F.col('StoricoShare_15_24_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_25_64_delta', F.col('StoricoShare_25_64') * F.col('StoricoShare') - F.col('StoricoShare_25_64_competitor') * F.col('StoricoShare_competitor'))
    .withColumn('StoricoShare_65_plus_delta', F.col('StoricoShare_65_plus') * F.col('StoricoShare') - F.col('StoricoShare_65_plus_competitor') * F.col('StoricoShare_competitor'))
)

df_programmi = (
    df_programmi.where(F.col('Canale').isin(['Rai 1', 'Rai 2', 'Rai 3','Rete 4','Canale 5','Italia 1','La7','Tv8','Nove']))
    .where(F.col('Ora') >= 7)
)

train_set = df_programmi.where(F.col('Data') <= '2026-03-31')
val_set = df_programmi.where((F.col('Data') > '2026-03-31') & (F.col('Data') <= '2026-04-30'))
test_set = df_programmi.where((F.col('Data') > '2026-04-30') & (F.col('Data') <= '2026-06-30'))

In [0]:
display(df_programmi)

In [0]:
print(f'Train size: {train_set.count()}')
print(f'Val size: {val_set.count()}')
print(f'Test size: {test_set.count()}')

# 2. Model Training

## 2.1 Feature Definition

In [0]:
categorical_features = [
    'Canale',
    'DES_GENERE_ESTESA_INT',
    'DES_GENERE_FILM_INT',
    'DES_GENERE_SPORT_INT',
    'DES_MANIFESTAZIONE_SPORT_INT',
    # 'DES_SPECIALITA_SPORT_INT',
    'Canale_competitor',
    'DES_GENERE_ESTESA_INT_competitor',
    'FlgPrimaVisione',
    'FlgPrimaVisioneGen',
    'FlgPrimaVisioneSpec',
    # 'Cod_Genere_Int',
    'Ora',
    'GiornoSettimana',
    'Mese',
]

categorical_features_ohe = {}

for col in categorical_features:
    col_values = train_set.select(col).distinct().orderBy(col).collect()
    categorical_features_ohe[col] = []

    for value in col_values:
        if value[0] is None:
            continue
        value_name = value[0].replace(' ', '_').replace('.', '_').replace('-','_')
        train_set = train_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))
        val_set = val_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))
        test_set = test_set.withColumn(f'{col}_{value_name}', F.when(F.col(col) == value[0], 1).otherwise(0))

        categorical_features_ohe[col].append(f'{col}_{value_name}')

train_set = train_set.toPandas()
val_set = val_set.toPandas()
test_set = test_set.toPandas()

In [0]:
X_columns = [
    'ORA_INIZIO_TRX',
    'Programma_durata',
    # 'Ora',
    # 'GiornoSettimana',
    # 'Mese',
    # 'StoricoShare',
    'StoricoShareMax',
    'StoricoShareMin',
    'StoricoShareLast',
    # 'StoricoShareStd',
    'StoricoShare_competitor',
    'StoricoShareLast_competitor',
    # 'StoricoShareUomini',
    # 'StoricoShareUomini_competitor',
    'StoricoShareUomini_delta',
    # 'StoricoShare_08_14',
    # 'StoricoShare_08_14_competitor',
    'StoricoShare_08_14_delta',
    # 'StoricoShare_15_24',
    # 'StoricoShare_15_24_competitor',
    'StoricoShare_15_24_delta',
    # 'StoricoShare_25_64',
    # 'StoricoShare_25_64_competitor',
    'StoricoShare_25_64_delta',
    # 'StoricoShare_65_plus',
    # 'StoricoShare_65_plus_competitor',
    'StoricoShare_65_plus_delta',
    # 'overlap_perc_competitor',
    'StoricoShare_prev',
    'StoricoShare_next',
    'StoricoShareLast_prev',
    'StoricoShareLast_next',
    'StoricoShareUomini_prev',
    'StoricoShareUomini_next',
    'HighValueShare',
    # 'LowValueShare',
    # 'StoricoEtaMedia',
    # 'Share',
]

for cat_ohe, values in categorical_features_ohe.items():
    X_columns.extend(values)

y_column = 'ShareResiduo'

X_train, y_train = train_set[X_columns], train_set[y_column]
X_val, y_val = val_set[X_columns], val_set[y_column]
X_test, y_test = test_set[X_columns], test_set[y_column]

## 2.2 Training

In [0]:
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from itertools import product
from sklearn.metrics import mean_absolute_error
import mlflow

try:
    mlflow.end_run()
except:
    pass
mlflow.start_run()

params_grid = {
    'n_estimators': [2000],
    'early_stopping_rounds': [10],
    'learning_rate': [0.01],
    'max_depth': [12],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_alpha': [0.1],
}

keys = params_grid.keys()
values = params_grid.values()

best_params = None
best_eval = float('inf')
best_model = None

for combo in product(*values):
    params = dict(zip(keys, combo))

    model = XGBRegressor(
        **params,
        monotone_constraints = {
            'StoricoShare_competitor': -1, 'StoricoShareLast_competitor': -1, 
            'StoricoShareUomini_delta': -1,
            'StoricoShare_08_14_delta': -1,
            'StoricoShare_15_24_delta': -1,
            'StoricoShare_25_64_delta': -1,
            'StoricoShare_65_plus_delta': -1,
            # 'StoricoShare_prev': 1, 'StoricoShare_next': 1, 'StoricoShareLast_prev': 1, 'StoricoShareLast_next': 1, 'StoricoShareLast': 1,
        },
        random_state=42,
    )

    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_val_pred = model.predict(X_val)

    eval = mean_absolute_error(y_val + val_set['StoricoShare'], y_val_pred + val_set['StoricoShare'])

    if eval < best_eval:
        best_eval = eval
        best_params = params
        best_model = model

    print(f'params: {params}, eval: {eval}')

model = best_model

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

mlflow.log_params(model.get_params())

mlflow.log_params({'X_columns': X_columns})
mlflow.log_params({'y_column': y_column})

In [0]:
mlflow.sklearn.log_model(model, 'model', input_example=X_test)

# 3. Output

## 3.1 Metrics

In [0]:
from sklearn.metrics import mean_absolute_error, r2_score

print('--- TRAIN SET ---')

_y_train = y_train + train_set['StoricoShare']
_y_train_pred = y_train_pred + train_set['StoricoShare']

mae = mean_absolute_error(_y_train, _y_train_pred)
print(f'MAE: {mae}')
r2 = r2_score(_y_train, _y_train_pred)
print(f'R2: {r2}')

mae_baseline = mean_absolute_error(_y_train, train_set['StoricoShare'])
print(f'MAE Baseline: {mae_baseline}')
r2_baseline = r2_score(_y_train, train_set['StoricoShare'])
print(f'R2 Baseline: {r2_baseline}')

mlflow.log_metrics({
    'mae_train': mae,
    'r2_train': r2,
    'mae_train_baseline': mae_baseline,
    'r2_train_baseline': r2_baseline,
})

print()
print('--- TEST SET ---')

_y_test = y_test + test_set['StoricoShare']
_y_test_pred = y_test_pred + test_set['StoricoShare']

mae = mean_absolute_error(_y_test, _y_test_pred)
print(f'MAE: {mae}')
r2 = r2_score(_y_test, _y_test_pred)
print(f'R2: {r2}')

mae_baseline = mean_absolute_error(_y_test, test_set['StoricoShare'])
print(f'MAE Baseline: {mae_baseline}')
r2_baseline = r2_score(_y_test, test_set['StoricoShare'])
print(f'R2 Baseline: {r2_baseline}')

mlflow.log_metrics({
    'mae_test': mae,
    'r2_test': r2,
    'mae_test_baseline': mae_baseline,
    'r2_test_baseline': r2_baseline,
})

mlflow.end_run()

## 3.2 Graphs

In [0]:
test_set['SharePrevisto'] = _y_test_pred
test_set['ErroreAssoluto'] = abs(_y_test_pred - _y_test)

In [0]:
display(spark.createDataFrame(test_set))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
import pandas as pd

# df = pd.DataFrame(list(zip(_y_test_pred, _y_test, test_set['StoricoShare'])), columns=['Prediction', 'True', 'Storico'])

display(spark.createDataFrame(pd.DataFrame(list(zip(_y_test_pred, _y_test, test_set['StoricoShare'], test_set['ShareResiduo'], y_test, y_test + train_set['StoricoShare'])), columns=['Prediction', 'True', 'Storico','ShareResiduo', 'ShareResiduoLog', 'ShareLog'])))

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
w = Window.orderBy("ORA_INIZIO_TRX")

df_step = (
    spark.createDataFrame(test_set)
    .where((F.col("Data") == "2026-01-10") & (F.col("Canale") == "Italia 1"))
    .withColumn("ORA_INIZIO_TRX", F.col("ORA_INIZIO_TRX") / 3600)
    .withColumn("ORA_FINE_TRX", F.col("ORA_FINE_TRX") / 3600)
    .withColumn("next_time", F.lead("ORA_INIZIO_TRX").over(w))
)

df_step = (
    df_step.select('Canale','Programma',"ORA_INIZIO_TRX", 'ORA_FINE_TRX', "Share", "SharePrevisto", 'StoricoShare', 'StoricoShare_prev')
    .unionByName(
        df_step.select(
            'Canale',
            'Programma',
            (F.col("next_time") - 0.0001).alias("ORA_INIZIO_TRX"),
            'ORA_FINE_TRX',
            "Share",
            "SharePrevisto",
            'StoricoShare',
            'StoricoShare_prev',
        ).where(F.col("next_time").isNotNull())
    ).withColumn('StoricoShare_prev', F.col('StoricoShare_prev') + F.col('StoricoShare'))
    .orderBy("ORA_INIZIO_TRX")
)

display(df_step)

Databricks visualization. Run in Databricks to view.

In [0]:
feature_importance = pd.DataFrame(
    zip(model.feature_names_in_, model.feature_importances_),
    columns=['column', 'importance']
)

spark.createDataFrame(feature_importance).display()

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.createDataFrame([(value,) for value in (_y_test_pred - _y_test)]).toDF('Errore'))

Databricks visualization. Run in Databricks to view.

In [0]:
display(spark.createDataFrame(test_set).orderBy(F.col('ErroreAssoluto').desc()))

Databricks visualization. Run in Databricks to view.

# 4. Inference

In [0]:
model_uri = 'runs:/8abf10c600ab4b318637f389a4910d6f/model'
loaded_model = mlflow.pyfunc.load_model(model_uri)
model.predict(pd.DataFrame(X_test))